# Controlled D1000 single-judge runtime renewal

Use only after a fully drained, independently readback-verified suspension. An arbitrary lost VM stays blocked because unsaved request outcomes may be unknown. Reuse the approved Restricted private notebook and its enabled secrets, or configure a new private copy yourself. CPU only. Do not Run all in the active writer notebook. The original20-hour deadline is retained. No training, inference, generation, paid fallback or quota reset.


In [ ]:
import subprocess,sys
subprocess.run([sys.executable,"-m","pip","install","-q","kaggle==2.2.5","groq==1.7.0","tiktoken==0.14.0","matplotlib","jsonschema"],check=True)


In [ ]:
# Use only a fresh private CPU VM after a controlled, readback-verified suspension.
import sys, json, hashlib, traceback
from pathlib import Path
if Path('/kaggle/working/d1000-authoritative-writer.lock').exists():
    raise RuntimeError('Existing writer VM: do not restore or replace it')
reviewed_sources = {'cloud/single_judge_fast/metrics.py': '"""Unchanged deterministic definitions and selected semantic votes; no agreement calls."""\nimport collections\nimport csv\nimport json\nfrom pathlib import Path\nimport statistics\nfrom cloud.single_judge_fast.protocol import PROTOCOL,select,rows\n\n\ndef summarise(root,votes):\n    from experiments import core\n    root=Path(root);out=root/\'results_single_judge_fast_v1\';out.mkdir(exist_ok=True)\n    ab=core.load_module(\'_fast_abstention\',\'eval/abstention.py\')\n    gold={g[\'id\']:g for g in rows(root/\'evaluation/gold.jsonl\')}\n    contexts={r[\'qid\']:r for r in rows(root/\'evaluation/contexts.jsonl\')}\n    cfg=json.loads((root/\'config.json\').read_text());chosen=select(votes);results=[]\n    mean=lambda xs:statistics.mean(x for x in xs if x is not None) if any(x is not None for x in xs) else None\n    for key,spec in cfg[\'models\'].items():\n        for condition in (\'base\',\'base-rag\',\'sft\',\'sft-rag\'):\n            path=root/\'runs\'/key/((\'sft\' if condition.startswith(\'sft\') else \'base\')+\'-responses.jsonl\')\n            responses={r[\'qid\']:r for r in rows(path) if r[\'condition\']==condition}\n            parsed=resolved=ncite=answerable_refused=unanswerable_refused=covered=0\n            correctness=[];support=[];citation=[];assign=collections.Counter();rag=condition.endswith(\'-rag\')\n            for q,g in gold.items():\n                r=responses.get(q)\n                if r is None:continue\n                answer=r[\'answer\'];refused=ab.is_abstention_assertion(answer,abstain_sentence=core.common.ABSTAIN_SENTENCE,citation_re=core.common.CITATION_RE)\n                check=core.rag.verify_citations(answer,contexts[q][\'chunks\']);parsed+=int(refused or check[\'n_total\']>0)\n                if g[\'answerable\']:answerable_refused+=int(refused)\n                else:unanswerable_refused+=int(refused)\n                if rag:resolved+=check[\'n_supported\'];ncite+=check[\'n_total\']\n                ident=core.digest({\'model\':r[\'model\'],\'condition\':condition,\'qid\':q})[:24]\n                if ident in chosen:\n                    provider,v=chosen[ident];covered+=1;assign[provider]+=1\n                    correctness.append(v[\'correctness\']);support.append(v[\'evidence_support\'])\n                    if rag:citation.extend(c[\'support\'] for c in v.get(\'citation_assessments\',[]))\n            available=len(responses)==76;refusals=answerable_refused+unanswerable_refused\n            row={\'protocol\':PROTOCOL,\'model\':spec[\'id\'],\'condition\':condition,\'available\':available,\n                 \'benchmark_questions\':76,\'responses\':len(responses),\'missing_responses\':76-len(responses),\n                 \'judged\':covered,\'missing_judgements\':76-covered if available else None,\n                 \'groq_selected\':assign[\'groq\'],\'gemini_selected\':assign[\'gemini\'],\n                 \'correctness\':mean(correctness),\'correctness_scored\':sum(x is not None for x in correctness),\n                 \'reference_support\':mean(support),\'citation_support\':mean(citation) if rag else None,\n                 \'parse_rate\':parsed/76 if available else None,\'citation_resolution\':resolved/ncite if rag and ncite else None,\n                 \'citation_denominator\':ncite if rag else None,\'answerable_denominator\':67,\'unanswerable_denominator\':9,\n                 \'appropriate_refusals\':unanswerable_refused if available else None,\n                 \'unnecessary_refusals\':answerable_refused if available else None,\n                 \'false_refusal_rate\':answerable_refused/67 if available else None,\n                 \'refusal_correctness\':unanswerable_refused/refusals if available and refusals else None,\n                 \'balanced_abstention\':.5*((67-answerable_refused)/67+unanswerable_refused/9) if available else None}\n            results.append(row)\n    (out/\'comparison.json\').write_text(json.dumps(results,indent=2))\n    with (out/\'comparison.csv\').open(\'w\',newline=\'\') as f:\n        writer=csv.DictWriter(f,fieldnames=list(results[0]));writer.writeheader();writer.writerows(results)\n    selected={ident:{\'provider\':p,\'request_sha256\':v[\'request_sha256\']} for ident,(p,v) in chosen.items()}\n    (out/\'selected_votes.json\').write_text(json.dumps(selected,sort_keys=True))\n    return {\'protocol\':PROTOCOL,\'unique_judged\':len(chosen),\'remaining\':912-len(chosen),\n            \'selected_provider_counts\':dict(collections.Counter(p for p,v in chosen.values())),\n            \'methodological_limitation\':\'Mixed-provider single valid judgement per response, Groq-first where both historical votes exist. Descriptive scores are not equivalent to the historical dual-judge design; no agreement analysis performed.\',\n            \'conditions\':16,\'deepseek\':\'unavailable\',\'full_coverage\':len(chosen)==912}\n', 'cloud/single_judge_fast/resume.py': '"""Restore only an independently verified, deliberately suspended fast checkpoint.\n\nNever recover an arbitrary lost VM as if its last request were unprocessed.\nRequires the same approved private Colab destination and supported existing secrets.\n"""\nfrom contextlib import redirect_stdout,redirect_stderr\nimport hashlib\nimport json\nimport os\nfrom pathlib import Path\nimport sys\nimport tempfile\nimport time\n\n\ndef validate_suspension(state,control,proof,version,now):\n    from cloud.single_judge_fast.lifecycle import clock\n    from cloud.single_judge_fast.protocol import PROTOCOL\n    if version<250 or state.get(\'status\')!=\'single_judge_suspended_for_supported_runtime_renewal\':\n        raise RuntimeError(\'Latest checkpoint is not a controlled suspension; reconcile unknown outcomes\')\n    if proof.get(\'protocol\')!=PROTOCOL or proof.get(\'phase\')!=\'suspended\' or proof.get(\'expected_checkpoint\')!=version:\n        raise RuntimeError(\'Suspension checkpoint identity differs\')\n    if proof.get(\'active_provider_threads\')!=0 or proof.get(\'active_provider_tasks\')!=0:\n        raise RuntimeError(\'Provider requests were not fully drained\')\n    if control.get(\'phase\')!=\'suspended\' or proof[\'launch_epoch\']!=control[\'launch_epoch\'] or proof[\'deadline_epoch\']!=control[\'deadline_epoch\']:\n        raise RuntimeError(\'Suspension clock differs\')\n    clock(control,now,proof[\'unique_judged\'])\n\n\ndef restore(runtime_scope,transport_scope,archive_audit,*,private_colab_transfer_approved=False):\n    if not private_colab_transfer_approved:raise PermissionError(\'Existing project approval required before private processing\')\n    if not Path(\'/content\').is_dir() or os.environ.get(\'COLAB_GPU\') not in (None,\'\',\'0\'):\n        raise RuntimeError(\'Private Colab CPU runtime required\')\n    if Path(\'/kaggle/working/d1000-authoritative-writer.lock\').exists() or Path(\'/kaggle/working/d1000\').exists():\n        raise RuntimeError(\'Use a fresh VM; do not replace an existing writer or restored state\')\n    from google.colab import userdata\n    from kaggle import api\n    for name in (\'KAGGLE_API_TOKEN\',\'GROQ_API_KEY\',\'GEMINI_API_KEY\'):\n        value=userdata.get(name)\n        if not isinstance(value,str) or not value:raise RuntimeError(\'Approved Colab secret unavailable\')\n        os.environ[name]=value;del value\n    self=runtime_scope[\'ColabRecovery\']()\n    Path(\'/kaggle/working\').mkdir(parents=True,exist_ok=True)\n    self.log=Path(\'/kaggle/working/fast-resume-private.log\');self.log.touch(mode=0o600);self.log.chmod(0o600)\n    with self.log.open(\'a\') as log,redirect_stdout(log),redirect_stderr(log):\n        api.authenticate();self.api=api\n        store=runtime_scope[\'STORE\'];kernel=runtime_scope[\'KERNEL\'];lease=runtime_scope[\'LEASE\']\n        if str(api.config_values.get(api.CONFIG_NAME_USER,\'\')).lower()!=store.split(\'/\')[0]:raise RuntimeError(\'Storage owner authentication required\')\n        if api.kernels_status(kernel).status.name not in runtime_scope[\'TERMINAL\']:raise RuntimeError(\'Kaggle writer active\')\n        with tempfile.TemporaryDirectory() as tmp:\n            api.kernels_pull(kernel+\'/18\',tmp,metadata=True)\n            notebooks=list(Path(tmp).glob(\'*.ipynb\'))\n            if len(notebooks)!=1:raise RuntimeError(\'Ambiguous audited source\')\n            files=runtime_scope[\'supported_bundle\'](json.loads(notebooks[0].read_text()))\n        code=Path(\'/kaggle/working/code\');runtime_scope[\'stage_bundle\'](code,files)\n        sys.path.insert(0,str(code));os.chdir(code)\n        from cloud.d1000 import evaluation_only as evaluation\n        from cloud.d1000.storage import info,publish,verify_file\n        self.info=info;self.publish=publish;self.evaluation=evaluation\n        audit=json.loads((code/\'cloud/d1000/evaluation_repair_v1.json\').read_text())\n        for p,h in {**audit[\'destination_hashes\'],**audit[\'immutable_code_hashes\']}.items():\n            if runtime_scope[\'sha\'](code/p)!=h:raise RuntimeError(\'Original source changed\')\n        for name,field in ((\'groq_probe_quota_deltas.json\',\'quota_deltas_sha256\'),(\'groq_calibration14_receipt.json\',\'groq_calibration_receipt_sha256\')):\n            if runtime_scope[\'sha\'](code/\'cloud/d1000\'/name)!=audit[field]:raise RuntimeError(\'Private recovery sideband changed\')\n        current=info(api,store)\n        if not current.is_private:raise RuntimeError(\'Results must remain private\')\n        version=current.current_version_number\n        mount=runtime_scope[\'INPUT_ROOT\']/(\'fast-resume\'+str(version));mount.mkdir(parents=True);mount.chmod(0o700)\n        for name in (\'progress.json\',\'artifacts.bin\'):\n            api.dataset_download_file(store+\'/\'+str(version),name,path=str(mount),quiet=True,force=True,unzip=True)\n            (mount/name).chmod(0o600)\n        saved=json.loads((mount/\'progress.json\').read_text())\n        if runtime_scope[\'sha\'](mount/\'artifacts.bin\')!=saved[\'artifacts_sha256\']:raise RuntimeError(\'Latest archive hash differs\')\n        scope=evaluation.__dict__.copy();exec(runtime_scope[\'restore_adapter\'](files[\'cloud/d1000/evaluation_only.py\']),scope)\n        state,preservation=scope[\'restore\']()\n        if preservation[\'immutable_manifest_sha256\']!=audit[\'immutable_manifest_sha256\']:raise RuntimeError(\'Frozen data changed\')\n        root=evaluation.ROOT;folder=root/\'single_judge_fast_v1\'\n        control=json.loads((folder/\'protocol.json\').read_text());proof=json.loads((folder/\'suspension.json\').read_text())\n        validate_suspension(state,control,proof,version,time.time())\n        self.state=state;self.preservation=preservation;self.receipt[\'preservation\']=preservation\n        # Audit the previously renewed receipt against its preserved original,\n        # never silently rewrite verification dates during a runtime restart.\n        self.receipt[\'provider_reverification\']=json.loads((root/\'recovery_audits/provider_reverification_v1/transition.json\').read_text())\n        transport_scope[\'prepare\'](self)\n        before=info(api,lease)\n        if not before.is_private or before.current_version_number!=proof[\'lease_version\']:raise RuntimeError(\'Prior lease changed\')\n        verify_file(lease,before.current_version_number,\'owner.json\',proof[\'lease_owner_sha256\'])\n        self.receipt[\'scratch_private_persistence\']=runtime_scope[\'scratch_persistence\'](api,info,verify_file,self.session)\n        if info(api,store).current_version_number!=version or info(api,lease).current_version_number!=before.current_version_number:\n            raise RuntimeError(\'Writer or source changed during renewal\')\n        with tempfile.TemporaryDirectory() as tmp:\n            owner=Path(tmp)/\'owner.json\';owner.write_text(json.dumps({\'session\':self.session,\'source_checkpoint\':239}))\n            result=publish(Path(tmp),lease,create=False)\n            self.lease_version=result[\'version\']\n        fd=os.open(\'/kaggle/working/d1000-authoritative-writer.lock\',os.O_CREAT|os.O_EXCL|os.O_WRONLY,0o600);os.close(fd)\n        self.ready=True;self.started=True;self.fast_handoff={\'old_workers_drained\':True,\'source_checkpoint\':version,\'prior_suspended_lease\':proof[\'lease_version\']}\n        self.fast_resume_control=control;self.fast_archive_audit=archive_audit\n        original=evaluation.checkpoint\n        def guarded_checkpoint(value):\n            lease_now=info(api,lease)\n            if not lease_now.is_private or lease_now.current_version_number!=self.lease_version:raise RuntimeError(\'Lease changed\')\n            if api.kernels_status(kernel).status.name not in runtime_scope[\'TERMINAL\']:raise RuntimeError(\'Concurrent Kaggle writer\')\n            return original(value)\n        evaluation.checkpoint=guarded_checkpoint\n        stop=folder/\'STOP_REQUESTED\'\n        if stop.exists():stop.rename(folder/(\'STOP_REQUESTED-consumed-at-checkpoint\'+str(version)))\n        from cloud.single_judge_fast import engine\n        report=engine.prepare(self)\n    return self,report\n', 'cloud/single_judge_fast/handoff.py': '"""Drain the old background worker after its control cell is interrupted at a checkpoint.\n\nNever interrupt provider threads, restart the VM, reset flags, or release the lease.\n"""\nimport json\nfrom pathlib import Path\nimport threading\nimport time\n\nPROTOCOL=\'d1000-single-judge-fast-v1\'\n\n\ndef old_workers():\n    return [t for t in threading.enumerate() if t.is_alive() and getattr(getattr(t,\'_target\',None),\'__name__\',\'\')==\'worker\']\n\n\ndef drain(recovery,sink,timeout=660):\n    from cloud.d1000 import judge_recovery\n    if not recovery.started or not recovery.ready or recovery.lease_version!=1:\n        raise RuntimeError(\'Existing verified writer required\')\n    previous=judge_recovery.complete\n    def no_new_calls(*args,**kwargs):\n        raise RuntimeError(\'Safe protocol handoff: no new old-protocol requests\')\n    judge_recovery.complete=no_new_calls\n    until=time.monotonic()+timeout\n    try:\n        while old_workers():\n            if time.monotonic()>until:raise RuntimeError(\'Old worker still active; no new writer allowed\')\n            for thread in old_workers():thread.join(timeout=2)\n        # Reached only after any submitted complete() returned and its worker\n        # journalled a result or protected uncertain outcome. No request is killed.\n        root=Path(recovery.evaluation.ROOT)\n        audit={\'protocol_transition_requested\':PROTOCOL,\'session_lease_reused\':True,\n               \'old_workers_drained\':True,\'transition_epoch\':time.time(),\n               \'prior_checkpoint\':recovery.state[\'last_persistence\'][\'version\']}\n        folder=root/\'recovery_audits\';folder.mkdir(exist_ok=True)\n        (folder/\'single_judge_handoff.json\').write_text(json.dumps(audit,sort_keys=True))\n        recovery.state[\'status\']=\'dual_judge_stopped_for_single_judge_handoff\'\n        recovery.evaluation.checkpoint(recovery.state)\n        audit[\'final_dual_checkpoint\']=recovery.state[\'last_persistence\'][\'version\']\n        sink({\'event\':\'old_coordinator_drained\',\'old_worker_count\':len(old_workers()),\n              \'same_lease_version\':recovery.lease_version,\'checkpoint\':audit[\'final_dual_checkpoint\'],\n              \'private_readback_verified\':recovery.state[\'last_persistence\'][\'readback_verified\']})\n        recovery.fast_handoff=audit\n        return audit\n    finally:\n        # If draining times out keep the stop gate installed; never revive calls.\n        if not old_workers():judge_recovery.complete=previous\n', 'cloud/single_judge_fast/protocol.py': '"""Single-valid-judge selection and read-only quota planning."""\nimport collections\nimport json\nfrom pathlib import Path\nimport statistics\nimport time\n\nPROTOCOL=\'d1000-single-judge-fast-v1\'\nMODELS={\'groq\':\'openai/gpt-oss-120b\',\'gemini\':\'gemini-3.5-flash-lite\'}\nUNKNOWN=(\'in_flight\',\'transport_uncertain\',\'response_saved\')\n\n\ndef rows(path):\n    return [json.loads(line) for line in Path(path).read_text().splitlines() if line.strip()] if Path(path).exists() else []\n\n\ndef select(votes):\n    """Deterministic Groq-first selection. Retain both source journals unchanged."""\n    selected={}\n    for provider in (\'gemini\',\'groq\'):\n        for ident,vote in votes[provider].items():selected[ident]=(provider,vote)\n    return selected\n\n\ndef load_validated(root,candidates):\n    from cloud.d1000 import judging,judge_recovery as jr\n    from experiments import core\n    root=Path(root);tasks={t[\'evaluation_id\']:t for t,_ in candidates};votes={p:{} for p in MODELS}\n    for provider in MODELS:\n        for vote in rows(root/\'judging\'/f\'{provider}.jsonl\'):\n            ident=vote[\'evaluation_id\']\n            if ident not in tasks or ident in votes[provider]:raise RuntimeError(\'Unexpected/duplicate vote\')\n            task=tasks[ident]\n            key=core.digest({\'task\':task,\'provider\':provider,\'model\':MODELS[provider],\'protocol\':jr.PROTOCOL})\n            cache=root/\'judge_runtime/evaluation_cache_v2\'/provider/(key+\'.json\')\n            record=json.loads(cache.read_text())\n            if record[\'protocol\']!=jr.PROTOCOL or record[\'input_sha256\']!=core.digest(task) or record[\'generation_config\'][\'model\']!=MODELS[provider]:\n                raise RuntimeError(\'Saved judgement identity mismatch\')\n            result={\'choices\':[{\'finish_reason\':\'stop\',\'message\':{\'content\':json.dumps({\'evaluations\':[record[\'indexed_answer\']]})}}]}\n            canonical=jr.validate(result,task,judging.provider_runtime(root)[0])\n            if canonical!=record[\'answer\'] or any(vote.get(k)!=v for k,v in canonical.items()):\n                raise RuntimeError(\'Journal and strictly validated cache differ\')\n            votes[provider][ident]=vote\n    return votes\n\n\ndef unknown_ids(root,candidates,provider):\n    from cloud.d1000 import judge_recovery as jr\n    from experiments import core\n    folder=Path(root)/\'judge_runtime/evaluation_cache_v2\'/provider;result=set()\n    for task,_ in candidates:\n        key=core.digest({\'task\':task,\'provider\':provider,\'model\':MODELS[provider],\'protocol\':jr.PROTOCOL})\n        path=folder/(key+\'.attempts.json\')\n        history=json.loads(path.read_text()) if path.exists() else []\n        if history and history[-1][\'status\'] in UNKNOWN:result.add(task[\'evaluation_id\'])\n    return result\n\n\ndef quota_audit(runtime,providers,now=None,hours=20):\n    from cloud.single_judge_fast.quotas import pacific_window\n    now=time.time() if now is None else now;result={}\n    for provider,model in MODELS.items():\n        limits=providers.verify_receipt(runtime,provider,model)\n        ledger=json.loads((Path(runtime)/\'provider_usage\'/f\'{provider}.json\').read_text())\n        current=[e for e in ledger[\'entries\'] if e[\'time\']>now-86400]\n        minute=[e for e in current if e[\'time\']>now-60]\n        usage=[providers.daily_tokens(e) for e in ledger[\'entries\'] if e.get(\'status\')==\'usage_recorded\']\n        remaining={\'rpm\':max(0,limits[\'rpm\']-len(minute)), \'rpd\':max(0,limits[\'rpd\']-len(current)),\n                   \'tpm\':max(0,limits[\'tpm\']-sum(e[\'quota_tokens\'] for e in minute)),\n                   \'tpd\':max(0,limits[\'tpd\']-sum(providers.daily_tokens(e) for e in current))}\n        expiring=[e for e in current if e[\'time\']+86401<=now+hours*3600]\n        slots=max(0,limits[\'rpd\']-len(current))+len(expiring)\n        tokens=max(0,limits[\'tpd\']-sum(providers.daily_tokens(e) for e in current))+sum(providers.daily_tokens(e) for e in expiring)\n        start,reset=pacific_window(now)\n        if provider==\'gemini\':\n            day=[e for e in ledger[\'entries\'] if e[\'time\']>=start]\n            remaining[\'rpd\']=max(0,limits[\'rpd\']-len(day))\n            slots=remaining[\'rpd\']+(limits[\'rpd\'] if reset+1<=now+hours*3600 else 0)\n        result[provider]={\'verified_limits\':{k:limits[k] for k in (\'rpm\',\'rpd\',\'tpm\',\'tpd\')},\n             \'remaining_conservative_ledger\':remaining,\'blocked_until_epoch\':ledger.get(\'blocked_until\',0),\n             \'next_minute_entry_expiry\':min((e[\'time\']+61 for e in minute),default=None),\n             \'next_daily_entry_expiry\':min((e[\'time\']+86401 for e in current),default=None),\n             \'twenty_hour_request_slots_upper_bound\':slots,\'twenty_hour_token_budget\':tokens,\n             \'historical_mean_actual_tokens\':statistics.mean(usage) if usage else None,\n             \'server_remaining_not_independently_known\':True,\n             \'documented_calendar_reset_epoch\':reset if provider==\'gemini\' else None,\n             \'quota_policy\':\'Gemini RPD midnight Pacific; RPM/TPM rolling60s and TPD rolling24h; all entries retained\' if provider==\'gemini\' else \'existing conservative rolling60s/24h; reservations unchanged\'}\n    return result\n\n\ndef feasibility(audit,remaining,token_estimates):\n    # Project with measured historical actual usage; do not equate projections to guarantees.\n    capacity={}\n    for p,q in audit.items():\n        mean=q[\'historical_mean_actual_tokens\']\n        cost=max(mean or 0,token_estimates[p][\'median_prompt_estimate\'])\n        capacity[p]=min(q[\'twenty_hour_request_slots_upper_bound\'],int(q[\'twenty_hour_token_budget\']/cost)) if cost else 0\n    return {\'remaining\':remaining,\'twenty_hour_projected_success_capacity_no_repairs\':capacity,\n            \'quota_projection_feasible\':sum(capacity.values())>=remaining,\n            \'request_slots_hard_upper_bound\':sum(q[\'twenty_hour_request_slots_upper_bound\'] for q in audit.values()),\n            \'interpretation\':\'Projection uses historical actual usage and current task estimates. Repairs, errors, shared account traffic and runtime limits can reduce coverage; not a completion promise.\'}\n', 'cloud/single_judge_fast/__init__.py': '"""Separate mixed-provider single-valid-judge protocol; original research unchanged."""\n', 'cloud/single_judge_fast/engine.py': '"""One writer, disjoint provider queues, original strict completion and replay guards."""\nimport asyncio\nimport hashlib\nimport json\nimport os\nfrom pathlib import Path\nimport statistics\nimport time\nfrom contextlib import redirect_stdout,redirect_stderr\nfrom cloud.single_judge_fast.protocol import PROTOCOL,MODELS,load_validated,unknown_ids,quota_audit,feasibility,select\nfrom cloud.single_judge_fast import metrics\nfrom cloud.single_judge_fast.handoff import old_workers\n\n\ndef writer_guard(recovery):\n    from cloud.d1000.storage import verify_file\n    if old_workers():raise RuntimeError(\'Historical worker still active\')\n    if not hasattr(recovery,\'fast_handoff\') or not recovery.started or not recovery.ready:\n        raise RuntimeError(\'Verified drained handoff required\')\n    if not Path(\'/kaggle/working/d1000-authoritative-writer.lock\').is_file():raise RuntimeError(\'Existing writer lock required\')\n    lease=\'scarletthe0116/uk-motor-d1000-evaluation-lease-v1\'\n    value=recovery.info(recovery.api,lease)\n    if not value.is_private or value.current_version_number!=recovery.lease_version:raise RuntimeError(\'Lease changed\')\n    owner=json.dumps({\'session\':recovery.session,\'source_checkpoint\':239}).encode()\n    verify_file(lease,recovery.lease_version,\'owner.json\',hashlib.sha256(owner).hexdigest())\n    if recovery.info(recovery.api,\'scarletthe0116/uk-motor-d1000-models-results\').current_version_number!=recovery.state[\'last_persistence\'][\'version\']:\n        raise RuntimeError(\'Concurrent checkpoint writer\')\n    if recovery.api.kernels_status(\'scarletthe0116/uk-motor-d1000-train-eval\').status.name not in (\'COMPLETE\',\'ERROR\',\'CANCEL_ACKNOWLEDGED\'):\n        raise RuntimeError(\'Active Kaggle writer\')\n    recovery.evaluation.verify_unchanged()\n\n\ndef prepare(recovery):\n    from cloud.d1000 import judging,judge_recovery as jr\n    writer_guard(recovery);root=Path(recovery.evaluation.ROOT)\n    providers,runtime=judging.provider_runtime(root)\n    from cloud.single_judge_fast.quotas import install\n    install(providers)\n    candidates=judging.tasks(root)\n    if len(candidates)!=912 or len({t[\'evaluation_id\'] for t,_ in candidates})!=912:raise RuntimeError(\'Population differs\')\n    votes=load_validated(root,candidates);covered=select(votes)\n    unknown={p:unknown_ids(root,candidates,p) for p in MODELS}\n    for p in MODELS:\n        calibration=json.loads((root/\'judging\'/f\'{p}-calibration-v2.json\').read_text());jr.verify_calibration(calibration)\n        catalogue,_=providers.transport({k:os.environ[k] for k in (\'GROQ_API_KEY\',\'GEMINI_API_KEY\')},p)\n        if not providers.catalogue_contains(p,MODELS[p],catalogue):raise RuntimeError(\'Exact model unavailable\')\n    costs={};estimates={}\n    for p in MODELS:\n        costs[p]={};prompts=[]\n        for task,_ in candidates:\n            if task[\'evaluation_id\'] in covered:continue\n            body=jr.body(p,MODELS[p],task)\n            counted=body[\'messages\']+[{\'role\':\'user\',\'content\':json.dumps(body[\'response_format\'])}]\n            reservation=providers.token_reservation(p,MODELS[p],counted,2048)\n            costs[p][task[\'evaluation_id\']]=reservation\n            # Planning estimate only. Actual calls still use exact original reservation.\n            prompts.append(max(1,reservation-2048) if p==\'groq\' else max(1,(reservation-6144)/4))\n        estimates[p]={\'median_prompt_estimate\':statistics.median(prompts) if prompts else 0,\n                      \'minimum_reservation\':min(costs[p].values(),default=0),\'maximum_reservation\':max(costs[p].values(),default=0)}\n    horizon=max(0,(recovery.fast_resume_control[\'deadline_epoch\']-time.time())/3600) if hasattr(recovery,\'fast_resume_control\') else 20\n    audit=quota_audit(runtime,providers,hours=horizon);forecast=feasibility(audit,912-len(covered),estimates)\n    report={\'protocol\':PROTOCOL,\'status\':\'READY_NOT_RUNNING\',\'source_checkpoint\':recovery.state[\'last_persistence\'][\'version\'],\n            \'unique_judged\':len(covered),\'remaining\':912-len(covered),\'provider_votes\':{p:len(votes[p]) for p in MODELS},\n            \'protected_task_counts\':{p:len(unknown[p]) for p in MODELS},\'quota_audit\':audit,\'token_estimates\':estimates,\n            \'feasibility\':forecast,\'selection\':\'Groq first if both valid; otherwise Gemini; never request a second vote\',\n            \'concurrency_per_provider\':1,\'judge_calls_in_preflight\':0}\n    recovery.fast_prepared={\'candidates\':candidates,\'votes\':votes,\'unknown\':unknown,\'costs\':costs,\'report\':report}\n    return report\n\n\ndef run(recovery,sink):\n    if getattr(recovery,\'fast_started\',False):raise RuntimeError(\'Fast coordinator already started\')\n    if not hasattr(recovery,\'fast_prepared\'):raise RuntimeError(\'Fast preflight required\')\n    writer_guard(recovery);recovery.fast_started=True\n    with recovery.log.open(\'a\') as log,redirect_stdout(log),redirect_stderr(log):\n        return asyncio.run(coordinate(recovery,sink))\n\n\nasync def run_async(recovery,sink,prelaunch_failure=None,ledger_hashes=None):\n    """Colab already owns its event loop; continue only an untouched loop-entry failure."""\n    if getattr(recovery,\'fast_started\',False):\n        if not isinstance(prelaunch_failure,RuntimeError) or str(prelaunch_failure)!=\'asyncio.run() cannot be called from a running event loop\':\n            raise RuntimeError(\'Fast coordinator already started\')\n        if hasattr(recovery,\'fast_control\'):\n            raise RuntimeError(\'Protocol already launched; cannot repeat entry\')\n        if not ledger_hashes or any(hashlib.sha256(Path(p).read_bytes()).hexdigest()!=v for p,v in ledger_hashes.items()):\n            raise RuntimeError(\'Quota state changed; reconcile before continuation\')\n    elif prelaunch_failure is not None:\n        raise RuntimeError(\'Unexpected prelaunch continuation\')\n    if not hasattr(recovery,\'fast_prepared\'):raise RuntimeError(\'Fast preflight required\')\n    writer_guard(recovery);recovery.fast_started=True\n    with recovery.log.open(\'a\') as log,redirect_stdout(log),redirect_stderr(log):\n        return await coordinate(recovery,sink)\n\n\nasync def coordinate(recovery,sink):\n    from cloud.d1000 import judging,judge_recovery as jr\n    from experiments import core\n    root=Path(recovery.evaluation.ROOT);folder=root/\'single_judge_fast_v1\';folder.mkdir(exist_ok=True)\n    prepared=recovery.fast_prepared;votes=prepared[\'votes\'];candidates=prepared[\'candidates\']\n    unknown=prepared[\'unknown\'];costs=prepared[\'costs\'];covered=set(select(votes));baseline=len(covered)\n    providers,runtime=judging.provider_runtime(root);values={k:os.environ[k] for k in (\'GROQ_API_KEY\',\'GEMINI_API_KEY\')}\n    from cloud.single_judge_fast.lifecycle import clock\n    prior=getattr(recovery,\'fast_resume_control\',None)\n    launched,deadline,baseline=clock(prior,time.time(),baseline)\n    control={\'protocol\':PROTOCOL,\'launch_epoch\':launched,\'deadline_epoch\':deadline,\'baseline_unique\':baseline,\n             \'source_checkpoint\':recovery.state[\'last_persistence\'][\'version\'],\'same_lease_version\':recovery.lease_version,\n             \'historical_protocol\':jr.PROTOCOL,\'selection_rule\':\'Groq-first historical tie; otherwise first valid provider\',\n             \'rubric_unchanged\':True,\'unknown_outcome_guard\':\'task/provider scoped; original complete() unchanged\',\n             \'source_hashes\':{p.name:hashlib.sha256(p.read_bytes()).hexdigest() for p in Path(__file__).parent.glob(\'*.py\')}}\n    if prior:\n        history=folder/\'history\';history.mkdir(exist_ok=True)\n        old=(folder/\'protocol.json\').read_bytes()\n        (history/(\'protocol-\'+hashlib.sha256(old).hexdigest()+\'.json\')).write_bytes(old)\n        control[\'resumed_from_checkpoint\']=recovery.state[\'last_persistence\'][\'version\']\n        control[\'original_source_checkpoint\']=prior.get(\'original_source_checkpoint\',prior[\'source_checkpoint\'])\n    (folder/\'protocol.json\').write_text(json.dumps(control,sort_keys=True,indent=2))\n    gate=asyncio.Event();gate.set();claims={};active=set();failed={p:set() for p in MODELS};status={p:{\'status\':\'ready\'} for p in MODELS}\n    recovery.fast_control=control;recovery.state[\'single_judge_fast\']=control\n    new_since_checkpoint=0;stopping=False\n    def append(path,value):\n        with path.open(\'a\') as f:f.write(json.dumps(value)+\'\\n\');f.flush();os.fsync(f.fileno())\n    def progress():\n        total=len(covered);elapsed=max(0,time.time()-launched);rate=(total-baseline)/(elapsed/3600) if elapsed else 0\n        audit=quota_audit(runtime,providers,hours=max(0,(deadline-time.time())/3600))\n        remaining=912-total;eta=remaining/rate if rate>0 else None\n        current_projected=0\n        for p,q in audit.items():\n            cost=max(q.get(\'historical_mean_actual_tokens\') or 0,prepared[\'report\'][\'token_estimates\'][p][\'median_prompt_estimate\'])\n            current_projected+=min(q[\'remaining_conservative_ledger\'][\'rpd\'],int(q[\'remaining_conservative_ledger\'][\'tpd\']/cost)) if cost else 0\n        reset_wait=max(0,audit.get(\'gemini\',{}).get(\'documented_calendar_reset_epoch\',time.time())-time.time())/3600 if audit else 0\n        if eta is not None and remaining>current_projected:\n            eta=max(eta,reset_wait+(remaining-current_projected)/rate)\n        return {\'protocol\':PROTOCOL,\'unique_judged\':total,\'remaining\':912-total,\n                \'provider_votes\':{p:len(votes[p]) for p in MODELS},\'new_unique_since_launch\':total-baseline,\n                \'successful_judgements_per_hour\':rate,\'estimated_hours_remaining\':eta,\n                \'eta_interpretation\':\'Measured successful unique throughput plus projected calendar quota wait; errors, repairs and runtime availability may worsen this estimate\',\n                \'elapsed_hours\':elapsed/3600,\'launch_epoch\':launched,\'deadline_epoch\':deadline,\n                \'provider_status\':status,\'unresolved_requests\':{p:len(unknown[p]) for p in MODELS},\n                \'quota_constraints\':audit,\'active_provider_calls\':len(active)}\n    async def checkpoint():\n        nonlocal new_since_checkpoint\n        gate.clear()\n        while active:await asyncio.sleep(.2)\n        report=progress();summary=metrics.summarise(root,votes)\n        report[\'selected_provider_counts\']=summary[\'selected_provider_counts\']\n        recovery.state[\'single_judge_fast\'].update(progress=report,metrics=summary)\n        recovery.state[\'status\']=\'single_judge_complete\' if len(covered)==912 else \'single_judge_partial\'\n        (folder/\'progress.json\').write_text(json.dumps(report,sort_keys=True,indent=2))\n        writer_guard(recovery)\n        # No provider calls or journal appends can race the snapshot.\n        await asyncio.to_thread(recovery.evaluation.checkpoint,recovery.state)\n        archive=Path(\'/kaggle/working/publish/artifacts.bin\')\n        stored=recovery.fast_archive_audit(archive)\n        expected={p:len(votes[p]) for p in MODELS}\n        if stored[\'persisted_judge_counts\']!=expected or any(stored[\'persisted_duplicate_votes\'].values()):raise RuntimeError(\'Snapshot vote journal mismatch\')\n        # Report end-to-end measured throughput including private publish/readback.\n        report=progress();report[\'selected_provider_counts\']=summary[\'selected_provider_counts\']\n        report.update(event=\'single_judge_private_checkpoint_readback\',checkpoint=recovery.state[\'last_persistence\'][\'version\'],\n                      private=True,readback_verified=True,archive_sha256=recovery.state[\'artifacts_sha256\'],persisted_judge_counts=expected)\n        sink(report);new_since_checkpoint=0;gate.set()\n    async def worker(p):\n        nonlocal new_since_checkpoint\n        model=MODELS[p]\n        while not stopping and len(covered)<912 and time.time()<deadline-900 and not (folder/\'STOP_REQUESTED\').exists():\n            await gate.wait()\n            if stopping:return\n            limits=providers.verify_receipt(runtime,p,model)\n            eligible=[t for t,m in candidates if t[\'evaluation_id\'] not in covered and t[\'evaluation_id\'] not in claims and t[\'evaluation_id\'] not in unknown[p] and t[\'evaluation_id\'] not in failed[p] and costs[p].get(t[\'evaluation_id\'],10**20)<=limits[\'tpm\']]\n            if not eligible:\n                status[p]={\'status\':\'no_safe_unclaimed_tasks\'};await asyncio.sleep(1);continue\n            task=min(eligible,key=lambda t:(costs[p][t[\'evaluation_id\']],t[\'evaluation_id\']))\n            ident=task[\'evaluation_id\'];claims[ident]=p;active.add(p)\n            try:\n                # Strict original body, validators, cache identity, reservations and\n                # durable raw-before-success journalling remain unchanged.\n                record=await asyncio.to_thread(jr.complete,values,p,model,task,runtime,providers,\n                        transport=lambda v,provider,actual:jr.observed_transport(v,provider,actual,runtime,providers))\n                if ident in covered:raise RuntimeError(\'Duplicate cross-provider assignment\')\n                row={**record[\'answer\'],\'provider\':p,\'model\':model,\'request_sha256\':record[\'request_sha256\'],\n                     \'usage\':record[\'usage\'],\'input_sha256\':record[\'input_sha256\'],\'evaluation_protocol\':PROTOCOL}\n                append(root/\'judging\'/f\'{p}.jsonl\',row);votes[p][ident]=row;covered.add(ident)\n                append(folder/\'new_assignments.jsonl\',{\'evaluation_id\':ident,\'provider\':p,\'request_sha256\':record[\'request_sha256\'],\'completed_epoch\':time.time()})\n                new_since_checkpoint+=1;status[p]={\'status\':\'running\',\'completed_votes\':len(votes[p])}\n            except providers.QuotaPause as exc:\n                status[p]={\'status\':\'quota_wait\',\'resume_after_epoch\':exc.until}\n            except Exception as exc:\n                reason=str(exc)\n                unknown[p].update(unknown_ids(root,candidates,p));failed[p].add(ident)\n                category=(\'unknown_outcome_protected\' if \'Unknown request outcome\' in reason else\n                          \'known_invalid_exhausted\' if \'Known-invalid judgement exhausted\' in reason else\n                          \'transport_uncertain_provider_stopped\' if ident in unknown[p] else \'provider_or_validation_failure\')\n                append(folder/\'safe_failures.jsonl\',{\'evaluation_id\':ident,\'provider\':p,\'category\':category,\'epoch\':time.time()})\n                status[p]={\'status\':\'blocked\' if category in (\'transport_uncertain_provider_stopped\',\'provider_or_validation_failure\') else \'task_skipped\',\'category\':category}\n                if status[p][\'status\']==\'blocked\':return\n            finally:\n                active.discard(p);claims.pop(ident,None)\n            until=status[p].get(\'resume_after_epoch\',0)\n            while time.time()<until and time.time()<deadline-900 and not stopping and not (folder/\'STOP_REQUESTED\').exists():\n                await asyncio.sleep(min(10,max(.2,until-time.time())))\n    await checkpoint() # protocol, metrics, lease, baseline persisted before new calls\n    workers=[asyncio.create_task(worker(p),name=\'fast-\'+p) for p in MODELS];last=time.monotonic()\n    try:\n        while len(covered)<912 and time.time()<deadline-900 and not (folder/\'STOP_REQUESTED\').exists():\n            if all(w.done() for w in workers):break\n            if not claims and all(status[p][\'status\'] in (\'blocked\',\'no_safe_unclaimed_tasks\') for p in MODELS):break\n            if new_since_checkpoint>=5 or time.monotonic()-last>=300:\n                await checkpoint();last=time.monotonic()\n            await asyncio.sleep(1)\n        stopping=True;gate.set()\n        # Drain calls naturally, including their original timeout and durable\n        # outcome handling; never cancel a running provider request.\n        await asyncio.gather(*workers)\n        await checkpoint()\n        return {\'status\':\'COMPLETE\' if len(covered)==912 else \'PARTIAL_STOPPED\',\'unique_judged\':len(covered),\n                \'remaining\':912-len(covered),\'checkpoint\':recovery.state[\'last_persistence\'][\'version\']}\n    except BaseException:\n        stopping=True;gate.set()\n        await asyncio.gather(*workers,return_exceptions=True)\n        raise\n    finally:\n        stopping=True;gate.set()\n', 'cloud/single_judge_fast/lifecycle.py': '"""A controlled, fully drained checkpoint is required before runtime renewal.\n\nAn arbitrary VM termination is NOT treated as evidence that unsaved requests were\nunprocessed. Such recovery stays blocked pending separate outcome reconciliation.\n"""\nimport asyncio\nimport hashlib\nimport json\nimport sys\nimport threading\nimport time\nfrom pathlib import Path\nfrom cloud.single_judge_fast.protocol import PROTOCOL,MODELS,select,load_validated,rows\n\n\ndef clock(prior,now,coverage):\n    if prior is None:return now,now+20*3600,coverage\n    if prior.get(\'protocol\')!=PROTOCOL or prior.get(\'phase\')!=\'suspended\':raise RuntimeError(\'Verified suspended protocol required\')\n    start=prior[\'launch_epoch\'];end=prior[\'deadline_epoch\']\n    if abs(end-start-72000)>.001:raise RuntimeError(\'Original deadline changed\')\n    if now>=end-900:raise RuntimeError(\'Insufficient time before original deadline\')\n    return start,end,prior[\'baseline_unique\']\n\n\ndef provider_threads():\n    matches=[]\n    for thread_id,frame in sys._current_frames().items():\n        while frame is not None:\n            name=frame.f_code.co_filename;function=frame.f_code.co_name\n            request=Path(name).name==\'judge_recovery.py\' and function in (\'complete\',\'observed_transport\')\n            persistence=any(part in name for part in (\'evaluation_only\',\'storage.py\',\'continue_startup\',\'original-audited-checkpoint\')) and function in (\'checkpoint\',\'checked_checkpoint\',\'publish\',\'verify_file\')\n            if request or persistence:\n                matches.append(thread_id);break\n            frame=frame.f_back\n    return matches\n\n\nasync def suspend(recovery,sink,timeout=660):\n    from cloud.d1000 import judging,judge_recovery as jr\n    from cloud.single_judge_fast import engine,metrics\n    from experiments import core\n    root=Path(recovery.evaluation.ROOT);folder=root/\'single_judge_fast_v1\'\n    if not getattr(recovery,\'fast_started\',False) or not hasattr(recovery,\'fast_control\'):raise RuntimeError(\'Existing fast coordinator required\')\n    (folder/\'STOP_REQUESTED\').touch(exist_ok=True)\n    until=time.monotonic()+timeout\n    while True:\n        workers=[t for t in asyncio.all_tasks() if t is not asyncio.current_task() and t.get_name() in (\'fast-groq\',\'fast-gemini\') and not t.done()]\n        if not workers and not provider_threads():break\n        if time.monotonic()>until:raise RuntimeError(\'Provider work still active; cannot suspend or renew runtime\')\n        await asyncio.sleep(1)\n    engine.writer_guard(recovery)\n    candidates=judging.tasks(root);providers,_=judging.provider_runtime(root)\n    # Preserve any fully validated cache success returned before an interrupted\n    # control cell could append its provider journal. Never issue an API call.\n    known={p:{v[\'evaluation_id\'] for v in rows(root/\'judging\'/f\'{p}.jsonl\')} for p in MODELS}\n    for task,_ in candidates:\n        for p,model in MODELS.items():\n            if task[\'evaluation_id\'] in known[p]:continue\n            key=core.digest({\'task\':task,\'provider\':p,\'model\':model,\'protocol\':jr.PROTOCOL})\n            cache=root/\'judge_runtime/evaluation_cache_v2\'/p/(key+\'.json\')\n            if not cache.exists():continue\n            record=json.loads(cache.read_text())\n            if record[\'protocol\']!=jr.PROTOCOL or record[\'input_sha256\']!=core.digest(task) or record[\'generation_config\'][\'model\']!=model:raise RuntimeError(\'Cached success identity differs\')\n            result={\'choices\':[{\'finish_reason\':\'stop\',\'message\':{\'content\':json.dumps({\'evaluations\':[record[\'indexed_answer\']]})}}]}\n            if jr.validate(result,task,providers)!=record[\'answer\']:raise RuntimeError(\'Cached result validation differs\')\n            value={**record[\'answer\'],\'provider\':p,\'model\':model,\'request_sha256\':record[\'request_sha256\'],\n                   \'usage\':record[\'usage\'],\'input_sha256\':record[\'input_sha256\'],\'evaluation_protocol\':PROTOCOL}\n            with (root/\'judging\'/f\'{p}.jsonl\').open(\'a\') as f:\n                f.write(json.dumps(value)+\'\\n\');f.flush();__import__(\'os\').fsync(f.fileno())\n            known[p].add(task[\'evaluation_id\'])\n    votes=load_validated(root,candidates);summary=metrics.summarise(root,votes)\n    owner=json.dumps({\'session\':recovery.session,\'source_checkpoint\':239}).encode()\n    proof={\'protocol\':PROTOCOL,\'phase\':\'suspended\',\'active_provider_threads\':0,\'active_provider_tasks\':0,\n           \'expected_checkpoint\':recovery.state[\'last_persistence\'][\'version\']+1,\n           \'lease_version\':recovery.lease_version,\'lease_owner_sha256\':hashlib.sha256(owner).hexdigest(),\n           \'suspended_epoch\':time.time(),\'unique_judged\':summary[\'unique_judged\'],\n           \'launch_epoch\':recovery.fast_control[\'launch_epoch\'],\'deadline_epoch\':recovery.fast_control[\'deadline_epoch\']}\n    recovery.fast_control.update(phase=\'suspended\',metrics=summary)\n    (folder/\'protocol.json\').write_text(json.dumps(recovery.fast_control,sort_keys=True,indent=2))\n    (folder/\'suspension.json\').write_text(json.dumps(proof,sort_keys=True,indent=2))\n    recovery.state[\'single_judge_fast\']=recovery.fast_control\n    recovery.state[\'status\']=\'single_judge_suspended_for_supported_runtime_renewal\'\n    await asyncio.to_thread(recovery.evaluation.checkpoint,recovery.state)\n    if recovery.state[\'last_persistence\'][\'version\']!=proof[\'expected_checkpoint\']:raise RuntimeError(\'Suspension checkpoint version changed\')\n    actual=recovery.fast_archive_audit(\'/kaggle/working/publish/artifacts.bin\')\n    if actual[\'persisted_judge_counts\']!={p:len(votes[p]) for p in MODELS} or any(actual[\'persisted_duplicate_votes\'].values()):raise RuntimeError(\'Suspension journal readback differs\')\n    recovery.fast_suspension=proof\n    sink({\'event\':\'single_judge_suspended_readback\',\'checkpoint\':proof[\'expected_checkpoint\'],\n          \'private_readback_verified\':True,\'unique_judged\':summary[\'unique_judged\'],\'remaining\':summary[\'remaining\'],\n          \'active_provider_threads\':0,\'active_provider_tasks\':0,\'original_deadline_epoch\':proof[\'deadline_epoch\']})\n    return proof\n', 'cloud/single_judge_fast/quotas.py': '"""Documented Gemini calendar-day RPD accounting without deleting ledger entries.\n\nSource: https://ai.google.dev/gemini-api/docs/rate-limits\nRPD resets midnight America/Los_Angeles; TPM/RPM and TPD remain conservative.\nThe original provider ledger, flock, atomic writes, reconciliation and pauses remain.\n"""\nfrom datetime import datetime,timedelta\nfrom zoneinfo import ZoneInfo\nimport time\n\n\ndef pacific_window(now):\n    local=datetime.fromtimestamp(now,ZoneInfo(\'America/Los_Angeles\'))\n    start=local.replace(hour=0,minute=0,second=0,microsecond=0)\n    return start.timestamp(),(start+timedelta(days=1)).timestamp()\n\n\ndef install(providers):\n    if hasattr(providers,\'fast_original_quotas\'):return\n    original=providers.Quotas\n    class DocumentedGeminiQuotas(original):\n        def reserve(self,ident,tokens,now=None):\n            if self.provider!=\'gemini\':return super().reserve(ident,tokens,now)\n            now=time.time() if now is None else now;start,reset=pacific_window(now)\n            if tokens>min(self.limits[\'tpm\'],self.limits[\'tpd\']):\n                raise providers.SafeAPIError(\'Request exceeds verified free token capacity; no truncation permitted\')\n            def action(state):\n                if state[\'blocked_until\']>now:raise providers.QuotaPause(self.provider,state[\'blocked_until\'])\n                minute=[e for e in state[\'entries\'] if e[\'time\']>now-60]\n                rolling=[e for e in state[\'entries\'] if e[\'time\']>now-86400]\n                day=[e for e in state[\'entries\'] if e[\'time\']>=start]\n                if len(minute)>=self.limits[\'rpm\'] or sum(e[\'quota_tokens\'] for e in minute)+tokens>self.limits[\'tpm\']:\n                    raise providers.QuotaPause(self.provider,min(e[\'time\'] for e in minute)+61)\n                if len(day)>=self.limits[\'rpd\']:raise providers.QuotaPause(self.provider,reset+1)\n                if sum(providers.daily_tokens(e) for e in rolling)+tokens>self.limits[\'tpd\']:\n                    raise providers.QuotaPause(self.provider,min(e[\'time\'] for e in rolling)+86401)\n                index=len(state[\'entries\'])\n                state[\'entries\'].append({\'request_sha256\':ident,\'time\':now,\'quota_tokens\':tokens,\'status\':\'reserved\'})\n                return index\n            return self.update(action)\n    providers.fast_original_quotas=original;providers.Quotas=DocumentedGeminiQuotas\n', 'cloud/recovery_v1/__init__.py': '"""Versioned evaluation-only runtime adapters; frozen D1000 source stays intact."""\n', 'cloud/recovery_v1/runtime.py': '"""Colab CPU adapter v1 for the exact supported D1000 evaluation source.\n\nNo research code is changed on disk. A narrowly checked restore adapter removes\nKaggle Secrets loading and the read-only input path; credentials use Colab userdata.\n"""\nimport ast\nfrom contextlib import redirect_stdout, redirect_stderr\nimport hashlib\nimport json\nimport os\nfrom pathlib import Path, PurePosixPath\nimport sys\nimport tempfile\nimport time\nimport uuid\n\nSTORE = \'scarletthe0116/uk-motor-d1000-models-results\'\nKERNEL = \'scarletthe0116/uk-motor-d1000-train-eval\'\nSOURCE_SHA = \'35e54432a06d06285699d8e9096cabd1bf2811693515934a092b1ea2c09d7133\'\nARCHIVE_SHA = \'aa9e5eba75d908c13da03c0c64ce47c55794e9aaeb74d16a25312fd90f86a604\'\nINPUT_ROOT = Path(\'/content/d1000-private-input\')\nSCRATCH = \'scarletthe0116/uk-motor-d1000-recovery-scratch-v1\'\nLEASE = \'scarletthe0116/uk-motor-d1000-evaluation-lease-v1\'\nTERMINAL = {\'COMPLETE\', \'ERROR\', \'CANCEL_ACKNOWLEDGED\'}\nCREDENTIAL_PROLOGUE = \'\'\'    from kaggle import api\n    from kaggle_secrets import UserSecretsClient\n    import os\n    secrets=UserSecretsClient()\n    for key in (\'KAGGLE_API_TOKEN\',\'GROQ_API_KEY\',\'GEMINI_API_KEY\'):\n        os.environ[key]=secrets.get_secret(key)\n    api.authenticate()\n\'\'\'\n\n\ndef sha(path):\n    h = hashlib.sha256()\n    with Path(path).open(\'rb\') as stream:\n        for chunk in iter(lambda: stream.read(1024*1024), b\'\'): h.update(chunk)\n    return h.hexdigest()\n\n\ndef supported_bundle(notebook):\n    """Read exact private code snapshot, not arbitrary downloaded executable code."""\n    source = \'\'.join(\'\'.join(c[\'source\']) for c in notebook[\'cells\'] if c[\'cell_type\'] == \'code\')\n    if hashlib.sha256(source.encode()).hexdigest() != SOURCE_SHA:\n        raise ValueError(\'Supported source hash mismatch\')\n    assignments = [n for n in ast.parse(source).body if isinstance(n, ast.Assign)\n                   and any(isinstance(t, ast.Name) and t.id == \'files\' for t in n.targets)]\n    if len(assignments) != 1: raise ValueError(\'Expected one frozen code bundle\')\n    files = ast.literal_eval(assignments[0].value)\n    for name, content in files.items():\n        p = PurePosixPath(name)\n        if p.is_absolute() or \'..\' in p.parts or not isinstance(content, str):\n            raise ValueError(\'Unsafe source bundle path\')\n        if p.parts[0] not in (\'experiments\', \'src\', \'eval\', \'cloud\', \'configs\', \'scripts\'):\n            raise ValueError(\'Unexpected source bundle path\')\n    return files\n\n\ndef stage_bundle(code, files):\n    """Reuse verified code after a preflight failure, never overwrite changed code."""\n    code = Path(code); code.mkdir(parents=True, exist_ok=True)\n    expected_py = {str(PurePosixPath(n)) for n in files if n.endswith(\'.py\')}\n    actual_py = {str(p.relative_to(code)) for p in code.rglob(\'*.py\')}\n    if actual_py - expected_py:\n        raise RuntimeError(\'Unexpected executable source in recovery directory\')\n    for name, text in files.items():\n        p = code/name\n        if p.exists() and (not p.is_file() or p.is_symlink() or p.read_text() != text):\n            raise RuntimeError(\'Existing recovery source differs from pinned bundle\')\n    for name, text in files.items():\n        p = code/name; p.parent.mkdir(parents=True, exist_ok=True)\n        if not p.exists(): p.write_text(text)\n        os.chmod(p,0o600)\n\n\ndef checkpoint_input(api, mount, name):\n    """Reuse only an exact hash-verified opaque archive; always refresh progress."""\n    if name not in (\'progress.json\', \'artifacts.bin\'):\n        raise ValueError(\'Unexpected checkpoint input\')\n    path = Path(mount)/name\n    if name == \'artifacts.bin\' and path.exists():\n        if sha(path) != ARCHIVE_SHA:\n            raise ValueError(\'Existing checkpoint archive differs from239\')\n        return\n    api.dataset_download_file(STORE+\'/239\', name, path=str(mount),\n                              quiet=True, force=True, unzip=True)\n\n\ndef scratch_persistence(api, info, verify_file, session, *, resume_created=False, attempts=18):\n    """One private create, then bounded readiness checks and exact nonce readback."""\n    if len(session) != 32 or any(c not in \'0123456789abcdef\' for c in session):\n        raise ValueError(\'Invalid scratch session\')\n    dataset = \'scarletthe0116/uk-motor-d1000-scratch-\' + session[:12]\n    if not resume_created:\n        with tempfile.TemporaryDirectory() as tmp:\n            folder = Path(tmp); (folder/\'nonce.txt\').write_text(session)\n            (folder/\'dataset-metadata.json\').write_text(json.dumps({\'id\':dataset,\n                \'title\':dataset.split(\'/\')[1], \'licenses\':[{\'name\':\'other\'}]}))\n            result = api.dataset_create_new(tmp,public=False,quiet=True,convert_to_csv=False,dir_mode=\'tar\')\n            if getattr(result,\'error\',None): raise RuntimeError(\'Private scratch creation rejected\')\n    # Acknowledged new resources may briefly return403 while provisioning. This\n    # never becomes an absence signal or another create attempt. Existing-resource\n    # recovery still requires private ownership and the exact original nonce.\n    for attempt in range(attempts):\n        try:\n            d=info(api,dataset)\n            if not d.is_private: raise RuntimeError(\'Scratch must remain private\')\n            if api.dataset_status(dataset)==\'ready\':\n                expected=hashlib.sha256(session.encode()).hexdigest()\n                verify_file(dataset,d.current_version_number,\'nonce.txt\',expected)\n                return {\'dataset\':dataset,\'version\':d.current_version_number,\'private\':True,\n                        \'readback_verified\':True,\'nonce_sha256\':expected}\n        except Exception as exc:\n            status=getattr(getattr(exc,\'response\',None),\'status_code\',None)\n            if status not in (403,404) or attempt==attempts-1: raise\n        if attempt+1<attempts: time.sleep(10)\n    raise RuntimeError(\'Scratch provisioning not verified; no judging allowed\')\n\n\ndef restore_adapter(source):\n    """Change credential acquisition only; all original restore checks retained."""\n    tree = ast.parse(source)\n    node = next(n for n in tree.body if isinstance(n, ast.FunctionDef) and n.name == \'restore\')\n    original = ast.get_source_segment(source, node) + \'\\n\'\n    if original.count(CREDENTIAL_PROLOGUE) != 1:\n        raise ValueError(\'Credential prologue differs from reviewed source\')\n    adapted = original.replace(CREDENTIAL_PROLOGUE,\n                               \'    from kaggle import api\\n    api.authenticate()\\n\', 1)\n    # Colab exposes /kaggle/input as read-only; use a private cloud directory.\n    if adapted.count("Path(\'/kaggle/input\')") != 1:\n        raise ValueError(\'Mounted-input expression differs from reviewed source\')\n    adapted = adapted.replace("Path(\'/kaggle/input\')", "Path(\'/content/d1000-private-input\')", 1)\n    # Only this function is compiled, under the unchanged module globals.\n    return compile(adapted, \'<reviewed-colab-restore-v1>\', \'exec\')\n\n\ndef rows(path):\n    return [json.loads(s) for s in Path(path).read_text().splitlines() if s.strip()]\n\n\ndef verify_journals(root):\n    """Validate saved counts and retain exact unknown-outcome protection."""\n    root = Path(root); votes = {}\n    for provider, count in ((\'groq\', 36), (\'gemini\', 345)):\n        entries = rows(root/\'judging\'/f\'{provider}.jsonl\')\n        ids = [r[\'evaluation_id\'] for r in entries]\n        if len(entries) != count or len(set(ids)) != count:\n            raise ValueError(\'Saved judgement population mismatch\')\n        votes[provider] = set(ids)\n    if len(votes[\'groq\'] & votes[\'gemini\']) != 32:\n        raise ValueError(\'Paired population mismatch\')\n    uncertain = {\'groq\': 0, \'gemini\': 0}\n    for provider in uncertain:\n        folder = root/\'judge_runtime/evaluation_cache_v2\'/provider\n        for path in folder.glob(\'*.attempts.json\'):\n            history = json.loads(path.read_text())\n            if not history or history[-1][\'status\'] not in (\'in_flight\',\'transport_uncertain\',\'response_saved\'):\n                continue\n            last = history[-1]; uncertain[provider] += 1\n            if provider == \'gemini\':\n                if last[\'status\'] != \'transport_uncertain\' or last.get(\'error_category\') != \'gemini HTTP 503\':\n                    raise ValueError(\'Unknown-outcome population changed\')\n                if (folder/(last[\'request_sha256\']+\'.raw.json\')).exists():\n                    raise ValueError(\'New durable reply requires separate reconciliation\')\n    if uncertain != {\'groq\': 0, \'gemini\': 1}:\n        raise ValueError(\'Expected protected Gemini uncertain request\')\n    # Hash all journals, caches and quota ledgers; this function never edits them.\n    retained = {str(p.relative_to(root)): sha(p) for directory in (\'judging\', \'judge_runtime\')\n                for p in (root/directory).rglob(\'*\') if p.is_file()}\n    return {\'groq\':36, \'gemini\':345, \'paired\':32, \'uncertain\':uncertain,\n            \'journals_manifest_sha256\':hashlib.sha256(json.dumps(retained,sort_keys=True).encode()).hexdigest()}\n\n\nclass ColabRecovery:\n    def __init__(self, *, scratch_session=None, resume_created_scratch=False):\n        self.resume_created_scratch = resume_created_scratch\n        self.ready = False; self.started = False\n        self.session = scratch_session or uuid.uuid4().hex\n        self.receipt = {\'adapter\':\'colab_cpu_v1_3\',\'status\':\'BLOCKED\',\'new_judge_calls\':0}\n\n    def prepare(self, *, private_colab_transfer_approved=False):\n        # This explicit gate comes before credential retrieval or private downloads.\n        if not private_colab_transfer_approved:\n            raise PermissionError(\'Confirm project permission for private checkpoint processing in your Colab VM\')\n        if not Path(\'/content\').is_dir(): raise RuntimeError(\'Managed Colab runtime required\')\n        from google.colab import userdata\n        if os.environ.get(\'COLAB_GPU\') not in (None, \'\', \'0\'):\n            raise RuntimeError(\'Select CPU runtime first\')\n        Path(\'/kaggle/working\').mkdir(parents=True, exist_ok=True)\n        os.chmod(\'/kaggle/working\', 0o700)\n        self.log = Path(\'/kaggle/working/recovery-private.log\')\n        self.log.touch(mode=0o600, exist_ok=True); os.chmod(self.log,0o600)\n        # Colab Secrets are local to this notebook/account; no Kaggle key transfer.\n        for key in (\'KAGGLE_API_TOKEN\', \'GROQ_API_KEY\', \'GEMINI_API_KEY\'):\n            value = userdata.get(key)\n            if not isinstance(value,str) or not value: raise RuntimeError(\'Required Colab secret unavailable\')\n            os.environ[key] = value\n            del value\n        with self.log.open(\'a\') as log, redirect_stdout(log), redirect_stderr(log):\n            self._prepare()\n        self.ready = True\n        self.receipt.update(status=\'READY_NOT_RUNNING\', judge_credentials_available=True,\n                            checkpoint=239, production_dataset_writes=0)\n        return self.receipt.copy()\n\n    def _prepare(self):\n        from kaggle import api\n        self.api = api; api.authenticate()\n        if str(api.config_values.get(api.CONFIG_NAME_USER, \'\')).lower() != STORE.split(\'/\')[0]:\n            raise RuntimeError(\'Results dataset owner authentication required\')\n        self.receipt[\'results_owner_authenticated\'] = True\n        # Pull only code/metadata in this private cloud VM; includes private audit sidebands.\n        if api.kernels_status(KERNEL).status.name not in TERMINAL:\n            raise RuntimeError(\'Kaggle evaluation writer is active\')\n        with tempfile.TemporaryDirectory() as tmp:\n            api.kernels_pull(KERNEL+\'/18\', tmp, metadata=True)\n            notebooks = list(Path(tmp).glob(\'*.ipynb\'))\n            if len(notebooks) != 1: raise RuntimeError(\'Ambiguous supported notebook\')\n            files = supported_bundle(json.loads(notebooks[0].read_text()))\n        code = Path(\'/kaggle/working/code\'); code.mkdir(exist_ok=True)\n        stage_bundle(code, files)\n        sys.path.insert(0,str(code)); os.chdir(code)\n        from cloud.d1000 import evaluation_only as evaluation\n        from cloud.d1000.storage import info, publish, verify_file\n        self.evaluation = evaluation\n        self.info = info; self.publish = publish\n        audit = json.loads((code/\'cloud/d1000/evaluation_repair_v1.json\').read_text())\n        for p,h in {**audit[\'destination_hashes\'], **audit[\'immutable_code_hashes\']}.items():\n            if sha(code/p) != h: raise RuntimeError(\'Audited source mismatch\')\n        # Private sideband hashes must match the existing repair audit.\n        for name, field in ((\'groq_probe_quota_deltas.json\',\'quota_deltas_sha256\'),\n                            (\'groq_calibration14_receipt.json\',\'groq_calibration_receipt_sha256\')):\n            if sha(code/\'cloud/d1000\'/name) != audit[field]: raise RuntimeError(\'Private audit sideband mismatch\')\n        d = info(api, STORE)\n        if not d.is_private or d.current_version_number != 239: raise RuntimeError(\'Reconcile latest checkpoint first\')\n        INPUT_ROOT.mkdir(parents=True, exist_ok=True); os.chmod(INPUT_ROOT,0o700)\n        mount = INPUT_ROOT/\'recovery239\'; mount.mkdir(exist_ok=True); os.chmod(mount,0o700)\n        for name in (\'progress.json\',\'artifacts.bin\'):\n            checkpoint_input(api,mount,name)\n        state = json.loads((mount/\'progress.json\').read_text())\n        if state[\'artifacts_sha256\'] != ARCHIVE_SHA or sha(mount/\'artifacts.bin\') != ARCHIVE_SHA:\n            raise RuntimeError(\'Checkpoint239 archive mismatch\')\n        # Original restore verifies latest private metadata, mounted exact progress,\n        # 912 responses, 76 questions, 3 adapters and unchanged config.\n        scope = evaluation.__dict__.copy()\n        exec(restore_adapter(files[\'cloud/d1000/evaluation_only.py\']),scope)\n        state, preservation = scope[\'restore\']()\n        if preservation[\'immutable_manifest_sha256\'] != audit[\'immutable_manifest_sha256\']:\n            raise RuntimeError(\'Frozen benchmark/model inventory mismatch\')\n        self.receipt[\'preservation\'] = preservation\n        self.receipt[\'saved_judgements\'] = verify_journals(evaluation.ROOT)\n        # Create-only scratch probe: a403 lookup is never reclassified as absence.\n        self.receipt[\'scratch_private_persistence\'] = scratch_persistence(api,info,verify_file,self.session,resume_created=self.resume_created_scratch)\n        if api.kernels_status(KERNEL).status.name not in TERMINAL or info(api,STORE).current_version_number != 239:\n            raise RuntimeError(\'Writer or checkpoint changed during preflight\')\n        self.state = state; self.preservation = preservation\n        # Bind only a reviewed restore adapter; original source files remain byte-identical.\n        evaluation.restore = lambda: (self.state,self.preservation)\n\n    def resume(self):\n        if not self.ready or self.started: raise RuntimeError(\'Successful preflight and one start required\')\n        api = self.api; evaluation = self.evaluation\n        if api.kernels_status(KERNEL).status.name not in TERMINAL:\n            raise RuntimeError(\'Another evaluation writer is active\')\n        if self.info(api,STORE).current_version_number != 239:\n            raise RuntimeError(\'Checkpoint changed; do not resume stale state\')\n        # A fixed private lease is created once. Existing lease => fail closed.\n        # On interrupted sessions, user must reconcile/release the lease before restart.\n        with self.log.open(\'a\') as log, redirect_stdout(log), redirect_stderr(log):\n            with tempfile.TemporaryDirectory() as tmp:\n                folder=Path(tmp); (folder/\'owner.json\').write_text(json.dumps({\'session\':self.session,\'source_checkpoint\':239}))\n                lease=self.publish(folder,LEASE,create=True)\n                self.lease_version=lease[\'version\']\n            fd=os.open(\'/kaggle/working/d1000-authoritative-writer.lock\',os.O_CREAT|os.O_EXCL|os.O_WRONLY,0o600)\n            os.close(fd); self.started=True\n            self.receipt.pop(\'new_judge_calls\', None)\n            original_checkpoint=evaluation.checkpoint\n            def checked_checkpoint(state):\n                d=self.info(api,LEASE)\n                if not d.is_private or d.current_version_number != self.lease_version:\n                    raise RuntimeError(\'Authoritative writer lease changed\')\n                if api.kernels_status(KERNEL).status.name not in TERMINAL:\n                    raise RuntimeError(\'Unexpected Kaggle writer\')\n                return original_checkpoint(state)\n            evaluation.checkpoint=checked_checkpoint\n            evaluation.main()  # Original checks, calibration, replay guards and 120s persistence.\n        # Report only aggregate data after original private write/readback.\n        receipt=json.loads(Path(\'/kaggle/working/evaluation_receipt.json\').read_text())\n        self.receipt.update(status=receipt[\'status\'],checkpoint=receipt[\'last_persistence\'][\'version\'],\n            judge_counts=receipt[\'judge_counts\'],paired_judgements=receipt.get(\'paired_judgements\'),\n            readback_verified=receipt[\'last_persistence\'][\'readback_verified\'],\n            new_persisted_judgements={p:receipt[\'judge_counts\'][p]-n for p,n in ((\'groq\',36),(\'gemini\',345))})\n        return self.receipt.copy()\n', 'cloud/recovery_v1/transport.py': '"""Verify archived transport against pinned production metadata; never reset ledgers."""\nimport hashlib\nimport importlib\nimport json\nfrom pathlib import Path\nimport sys\nfrom contextlib import redirect_stdout,redirect_stderr\n\nPRODUCTION=\'scarletthe0116/uk-motor-qa-production-v1\'\nARCHIVE=\'a88e3259cde5357f273f97a9ff2afa0821359815a7238c78332056d16511fe0e\'\n\n\ndef digest(path):return hashlib.sha256(Path(path).read_bytes()).hexdigest()\n\n\ndef install_verified(judging, root, manifest, receipt_transition=None):\n    if manifest.get(\'archive_sha256\')!=ARCHIVE:\n        raise RuntimeError(\'Original production archive identity differs\')\n    root=Path(root);code,runtime=judging.transport_paths(root)\n    expected={code/name:manifest[\'files\'][\'uk-insurance-QA-data/src/\'+name]\n              for name in (\'providers.py\',\'common.py\',\'kelly.py\')}\n    expected[runtime/\'provider_verification.json\']=manifest[\'files\'][\'uk-insurance-QA-data-local/production-v1/provider_verification.json\']\n    def provider_runtime(actual_root):\n        if Path(actual_root).resolve()!=root.resolve():raise RuntimeError(\'Unexpected transport root\')\n        for path,value in expected.items():\n            if path==runtime/\'provider_verification.json\' and receipt_transition is not None:\n                from cloud.recovery_v1.reverify import validate_transition\n                validate_transition(path,value,receipt_transition)\n                continue\n            if not path.is_file() or digest(path)!=value:raise RuntimeError(\'Archived transport hash mismatch\')\n        ledgers=[runtime/\'provider_usage\'/f\'{p}.json\' for p in (\'groq\',\'gemini\')]\n        if not all(p.is_file() for p in ledgers):raise RuntimeError(\'Existing quota ledgers required\')\n        before={p:digest(p) for p in ledgers}\n        sys.path.insert(0,str(code.parent))\n        providers=importlib.import_module(\'production_transport.providers\')\n        if Path(providers.__file__).resolve()!=(code/\'providers.py\').resolve():\n            raise RuntimeError(\'Unexpected transport import location\')\n        if before!={p:digest(p) for p in ledgers}:raise RuntimeError(\'Quota ledger changed during transport loading\')\n        return providers,runtime\n    # Verify before installing; no code, receipts or ledgers are written.\n    provider_runtime(root)\n    judging.provider_runtime=provider_runtime\n    return {\'transport_hashes_verified\':len(expected),\'quota_ledgers_preserved\':True,\n            \'source_production_version\':319,\'source_archive_sha256\':ARCHIVE}\n\n\ndef prepare(recovery):\n    from cloud.d1000 import judging\n    from cloud.d1000.storage import info\n    folder=Path(\'/content/d1000-private-input/transport319\');folder.mkdir(parents=True,exist_ok=True)\n    folder.chmod(0o700)\n    with recovery.log.open(\'a\') as log,redirect_stdout(log),redirect_stderr(log):\n        if not info(recovery.api,PRODUCTION).is_private:raise RuntimeError(\'Production source must remain private\')\n        recovery.api.dataset_download_file(PRODUCTION+\'/319\',\'migration_manifest.json\',\n            path=str(folder),quiet=True,force=True,unzip=True)\n    path=folder/\'migration_manifest.json\';path.chmod(0o600)\n    result=install_verified(judging,recovery.evaluation.ROOT,json.loads(path.read_text()),recovery.receipt.get(\'provider_reverification\'))\n    providers,runtime=judging.provider_runtime(recovery.evaluation.ROOT)\n    for provider,model in ((\'groq\',\'openai/gpt-oss-120b\'),(\'gemini\',\'gemini-3.5-flash-lite\')):\n        providers.verify_receipt(runtime,provider,model)\n    recovery.receipt[\'archived_transport\']=result\n    return result\n', 'cloud/recovery_v1/reverify.py': '"""Renew account verification only after actual current checks; never renew quotas."""\nimport hashlib\nimport json\nimport os\nfrom pathlib import Path\nimport time\n\nMODELS={\'groq\':\'openai/gpt-oss-120b\',\'gemini\':\'gemini-3.5-flash-lite\'}\nREQUIRED=(\'free_plan_confirmed\',\'model_catalogue_verified\',\'synthetic_json_probe_passed\',\n          \'policy_transfer_authorised\',\'data_use_terms_compatible\',\'training_dataset_use_compatible\')\n\n\ndef sha(path):return hashlib.sha256(Path(path).read_bytes()).hexdigest()\n\n\ndef renew(recovery, providers, runtime, verify_journals, account_evidence):\n    from cloud.d1000.judge_recovery import verify_calibration,PROTOCOL\n    from eval.semantic_fixtures import fixtures\n    from experiments import core\n    root=Path(recovery.evaluation.ROOT);runtime=Path(runtime)\n    before=verify_journals(root)\n    if before!=recovery.receipt[\'saved_judgements\']:raise RuntimeError(\'Journal state changed\')\n    now=time.time()\n    if set(account_evidence)!=set(MODELS):raise RuntimeError(\'Both account checks required\')\n    path=runtime/\'provider_verification.json\';original=path.read_bytes();old=json.loads(original)\n    retained={p:sha(p) for directory in (\'judging\',\'judge_runtime\') for p in (root/directory).rglob(\'*\') if p.is_file() and p!=path}\n    checks={}\n    for provider,model in MODELS.items():\n        evidence=account_evidence[provider]\n        if evidence.get(\'free_plan_observed\') is not True or evidence.get(\'terms_reviewed\') is not True:\n            raise RuntimeError(\'Current free account and data-use checks required\')\n        if not 0<=now-evidence.get(\'observed_at_epoch\',0)<=3600:raise RuntimeError(\'Current account evidence required\')\n        item=old[provider]\n        if item.get(\'model\')!=model or not all(item.get(k) is True for k in REQUIRED):\n            raise RuntimeError(\'Cannot create a missing original attestation\')\n        catalogue,_=providers.transport({k:os.environ[k] for k in (\'GROQ_API_KEY\',\'GEMINI_API_KEY\')},provider)\n        if not providers.catalogue_contains(provider,model,catalogue):raise RuntimeError(\'Pinned model absent\')\n        calibration_path=root/\'judging\'/f\'{provider}-calibration-v2.json\'\n        calibration=json.loads(calibration_path.read_text());verify_calibration(calibration)\n        if not (calibration[\'passed\'] and calibration[\'protocol\']==PROTOCOL and calibration[\'fixtures_sha256\']==core.digest(fixtures())):\n            raise RuntimeError(\'Original calibration cannot be reused\')\n        checks[provider]={\'account\':evidence,\'model\':model,\'catalogue_verified\':True,\'original_calibration_sha256\':sha(calibration_path),\'inference_calls\':0}\n    # No cache, vote, reservation, limit, permission or methodology changes.\n    current={p:sha(p) for directory in (\'judging\',\'judge_runtime\') for p in (root/directory).rglob(\'*\') if p.is_file() and p!=path}\n    if current!=retained:raise RuntimeError(\'Request state changed during verification\')\n    new=json.loads(original)\n    for provider in MODELS:new[provider][\'verified_at_epoch\']=now\n    encoded=json.dumps(new,sort_keys=True,indent=2).encode()\n    audit=root/\'recovery_audits\'/\'provider_reverification_v1\';audit.mkdir(parents=True,exist_ok=False);audit.chmod(0o700)\n    backup=audit/\'original_provider_verification.json\';backup.write_bytes(original);backup.chmod(0o600)\n    path.write_bytes(encoded);path.chmod(0o600)\n    try:\n        for provider,model in MODELS.items():providers.verify_receipt(runtime,provider,model)\n    except BaseException:\n        path.write_bytes(original);raise\n    transition={\'original_sha256\':hashlib.sha256(original).hexdigest(),\'renewed_sha256\':sha(path),\n                \'backup\':str(backup),\'checks\':checks,\'quota_limits_unchanged\':True,\'request_state_unchanged\':True}\n    (audit/\'transition.json\').write_text(json.dumps(transition,sort_keys=True,indent=2))\n    (audit/\'transition.json\').chmod(0o600)\n    recovery.receipt[\'saved_judgements\']=verify_journals(root)\n    recovery.receipt[\'provider_reverification\']=transition\n    return transition\n\n\ndef validate_transition(path,original_hash,transition):\n    if transition.get(\'original_sha256\')!=original_hash or sha(transition[\'backup\'])!=original_hash or sha(path)!=transition.get(\'renewed_sha256\'):\n        raise RuntimeError(\'Provider receipt transition identity differs\')\n    old=json.loads(Path(transition[\'backup\']).read_text());new=json.loads(Path(path).read_text())\n    for provider in MODELS:\n        if new[provider].get(\'verified_at_epoch\',0)<=old[provider].get(\'verified_at_epoch\',0):raise RuntimeError(\'No current verification\')\n        new[provider][\'verified_at_epoch\']=old[provider][\'verified_at_epoch\']\n        check=transition[\'checks\'][provider]\n        if check.get(\'catalogue_verified\') is not True or check.get(\'inference_calls\')!=0:raise RuntimeError(\'Missing verification evidence\')\n    if old!=new or not transition.get(\'request_state_unchanged\') or not transition.get(\'quota_limits_unchanged\'):\n        raise RuntimeError(\'Receipt fields other than verified time changed\')\n', 'cloud/recovery_v1/monitor.py': '"""Aggregate-only notebook progress after original checkpoint readback."""\ndef safe_progress(state):\n    persisted = state[\'last_persistence\']\n    if persisted.get(\'private\') is not True or persisted.get(\'readback_verified\') is not True:\n        raise RuntimeError(\'Checkpoint readback must complete before reporting success\')\n    counts = state[\'judge_counts\']\n    return {\'event\':\'private_checkpoint_readback\',\'checkpoint\':persisted[\'version\'],\n            \'private\':True,\'readback_verified\':True,\'archive_sha256\':state[\'artifacts_sha256\'],\n            \'judge_counts\':{p:counts[p] for p in (\'groq\',\'gemini\')},\n            \'responses_preserved\':state[\'responses_completed\'],\n            \'providers\':{p:{k:v for k,v in value.items() if k in\n                (\'status\',\'completed\',\'calibration\',\'resume_after_epoch\')}\n                for p,value in state.get(\'judges\',{}).items()}}\n\n\ndef install(evaluation, sink):\n    original = evaluation.checkpoint\n    def observed_checkpoint(state):\n        result = original(state)\n        report=safe_progress(state)\n        from pathlib import Path\n        archive=Path(\'/kaggle/working/publish/artifacts.bin\')\n        if archive.is_file():report.update(saved_vote_counts(archive))\n        sink(report)\n        return result\n    evaluation.checkpoint = observed_checkpoint\n\n\ndef saved_vote_counts(archive):\n    """Read only vote IDs from the just-readback archive inside the cloud VM."""\n    import json\n    import tarfile\n    votes={};duplicates={}\n    wanted={\'judging/groq.jsonl\':\'groq\',\'judging/gemini.jsonl\':\'gemini\'}\n    with tarfile.open(archive,mode=\'r|gz\') as snapshot:\n        for member in snapshot:\n            if member.name not in wanted:continue\n            provider=wanted[member.name]\n            rows=[json.loads(line) for line in snapshot.extractfile(member) if line.strip()]\n            ids={row[\'evaluation_id\'] for row in rows}\n            votes[provider]=ids;duplicates[provider]=len(rows)-len(ids)\n            if len(votes)==2:break\n    if set(votes)!={\'groq\',\'gemini\'}:raise RuntimeError(\'Persisted vote journals absent\')\n    return {\'persisted_judge_counts\':{p:len(votes[p]) for p in votes},\n            \'persisted_paired\':len(votes[\'groq\'] & votes[\'gemini\']),\n            \'persisted_duplicate_votes\':duplicates,\'persisted_vote_journals_audited\':True}\n', 'cloud/single_judge_fast/final_report.py': '"""Read aggregate tables only after a normal, readback-verified coordinator finish."""\nimport json\nfrom pathlib import Path\nfrom cloud.single_judge_fast.protocol import PROTOCOL\n\n\ndef report(root,result,state):\n    if result.get(\'status\') not in (\'COMPLETE\',\'PARTIAL_STOPPED\'):\n        raise RuntimeError(\'Coordinator did not finish through a verified checkpoint\')\n    persistence=state[\'last_persistence\']\n    if not persistence.get(\'private\') or not persistence.get(\'readback_verified\') or result[\'checkpoint\']!=persistence[\'version\']:\n        raise RuntimeError(\'Final checkpoint readback not verified\')\n    table=json.loads((Path(root)/\'results_single_judge_fast_v1/comparison.json\').read_text())\n    if len(table)!=16 or any(r[\'protocol\']!=PROTOCOL or r[\'benchmark_questions\']!=76 for r in table):\n        raise RuntimeError(\'Comparison population differs\')\n    covered=sum(r[\'judged\'] for r in table)\n    if covered!=result[\'unique_judged\'] or covered+result[\'remaining\']!=912:\n        raise RuntimeError(\'Comparison coverage differs from verified result\')\n    unavailable=[r for r in table if not r[\'available\']]\n    if len(unavailable)!=4 or any(r[\'responses\']!=0 or r[\'correctness\'] is not None for r in unavailable):\n        raise RuntimeError(\'Unavailable model conditions must remain N/A\')\n    return {\'protocol\':PROTOCOL,\'complete\':covered==912,\'unique_judged\':covered,\'remaining\':912-covered,\n            \'checkpoint\':persistence[\'version\'],\'private_readback_verified\':True,\n            \'comparison\':table,\n            \'methodological_limitation\':\'Mixed-provider single valid judgement, deterministic Groq-first historical tie rule; not dual-judge equivalence or inter-judge agreement. Missing semantic votes are not zero; unavailable DeepSeek conditions remain N/A.\'}\n'}
reviewed_hashes = {'cloud/single_judge_fast/metrics.py': '3c03a50d6121e7aee965d8ec773d13b6b111179b2422b6d1bfb883dae6aac8a9', 'cloud/single_judge_fast/resume.py': '1185c067de51f01f1c4d82279f4fc5b6556b78bbae06d652f6b4e915b1968857', 'cloud/single_judge_fast/handoff.py': '7e727b2d7cfddd5c34e324254c78dfacd9dc176a771813ab2316d0619eee9d5e', 'cloud/single_judge_fast/protocol.py': '6f17f72dc258ab65271115988b3036000f4e4bde93a97d7c09333883c9bb0a74', 'cloud/single_judge_fast/__init__.py': '3c3e01db12655da6fd015305e8b0e16e615b7cb4cfad909296667259fa138a0b', 'cloud/single_judge_fast/engine.py': '7be885033cc1d60d827f3e8ec92a2442861da342b9262e98445e2975c57b7a9b', 'cloud/single_judge_fast/lifecycle.py': '10823b428be2941bb91322c427314d3348c8c90e661f8f02ae276ee414879fe5', 'cloud/single_judge_fast/quotas.py': 'f20ddaa5759ce4251a0d916898d4e767581deb9f856f37aff08ffc9c41e5e7f0', 'cloud/recovery_v1/__init__.py': 'cfd81184c8e2ba1bafdd2646c68b4a5d987f55a47f238e3718d2151c0f681f55', 'cloud/recovery_v1/runtime.py': '14e840ffd2475f1c8f9ee55f08d9eb1fe6ea0faf5cf56dea23f4296b8aa1f9f4', 'cloud/recovery_v1/transport.py': 'fd9ee508cba740d2a85c1a8990c016bfdc53a57088bfaa5be088f1af53955002', 'cloud/recovery_v1/reverify.py': 'f543938d45375368127492273dda70e1b8a3dcb5f377e8e0a0e8b8a623e606ac', 'cloud/recovery_v1/monitor.py': '459c4f714f0482de0e065a615255138403f30ed40630f8e1fb3a0fb5d064f7bb', 'cloud/single_judge_fast/final_report.py': '561549e9596c069747ec9364860eab86b93536062917656502d5feaf872a7074'}
helper_root=Path('/content/d1000-fast-reviewed-helpers')
for name,source in reviewed_sources.items():
    assert hashlib.sha256(source.encode()).hexdigest()==reviewed_hashes[name]
    target=helper_root/name
    target.parent.mkdir(parents=True,exist_ok=True)
    if target.exists():assert target.read_text()==source, 'Existing helper source differs'
    target.write_text(source)
sys.path.insert(0,str(helper_root))
from cloud.recovery_v1 import runtime,transport,monitor
from cloud.single_judge_fast import resume,engine
try:
    # The owner approved this exact private Colab processing on1 November2026.
    recovery,preflight=resume.restore(vars(runtime),vars(transport),monitor.saved_vote_counts,
        private_colab_transfer_approved=True)
    print(json.dumps(preflight))
except BaseException as exc:
    resume_failure=exc
    print(json.dumps({'status':'BLOCKED','stage':'controlled_fast_runtime_restore',
        'error_type':type(exc).__name__,'frames':[(Path(f.filename).name,f.name,f.lineno)
        for f in traceback.extract_tb(exc.__traceback__)]}))


In [ ]:
# Requires the controlled restore above to pass. Do not run concurrently elsewhere.
import sys,json,traceback
out=sys.stdout
def checkpoint_report(value):
    out.write(json.dumps(value)+'\n');out.flush()
try:
    print(json.dumps({'event':'controlled_resume_requested','original_deadline_epoch':recovery.fast_resume_control['deadline_epoch']}))
    result=await engine.run_async(recovery,checkpoint_report)
    print(json.dumps(result))
    from cloud.single_judge_fast.final_report import report
    print(json.dumps(report(recovery.evaluation.ROOT,result,recovery.state)))
except BaseException as exc:
    print(json.dumps({'status':'BLOCKED','error_type':type(exc).__name__,
        'frames':[(Path(f.filename).name,f.name,f.lineno) for f in traceback.extract_tb(exc.__traceback__)]}))
